# Notebook 9 — Agent Protocol Stack : MCP, A2A, AG-UI
### BNP Paribas — Senior AI Engineer

---
## Sommaire
1. [Qu’est-ce qu’un agent IA ?](#agent)
2. [Patterns d’agence : ReAct, Plan-and-Execute, Reflexion](#patterns)
3. [Tool Calling & Function Calling](#tools)
4. [MCP — Model Context Protocol](#mcp)
5. [A2A — Agent-to-Agent Protocol](#a2a)
6. [AG-UI — Agent-User Interaction Protocol](#agui)
7. [Orchestration multi-agents](#orchestration)
8. [Evaluation et fiabilite des agents](#eval)
9. [Questions d’entretien](#questions)


---
## 1. Qu’est-ce qu’un agent IA ? <a name='agent'></a>

### Definition
Un agent IA est un LLM capable de :
1. **Percevoir** son environnement (inputs : texte, outils, memoire)
2. **Planifier** une sequence d'actions
3. **Agir** via des outils (fonctions, APIs, code)
4. **Observer** le resultat et iterer

### Composants d'un agent
```
+-----------+      +-----------+      +------------+
|  LLM Core |<---> |  Memory   |      |  Tools     |
|  (Cerveau)|      | (ST / LT) |      | (Fonctions)|
+-----------+      +-----------+      +------------+
      |                                     |
      +-------------------------------------+
      |          Agent Loop                 |
      v                                     v
  Observation <---- Action <---- Decider ---+
```

### Types de memoire
| Type | Description | Exemple |
|---|---|---|
| Short-term (in-context) | Historique de la conversation | Messages precedents |
| Long-term (external) | Base de connaissances persistante | VDB, SQL |
| Episodic | Experiences passees de l agent | Logs d execution |
| Semantic | Connaissances generales | Parametres du LLM |
| Working memory | Etat intermediaire de la tache | Scratchpad |

### Niveaux d'autonomie
| Niveau | Description | Controle humain |
|---|---|---|
| 0 | LLM seul | Maximal |
| 1 | LLM + outils | Eleve |
| 2 | Agent avec boucle | Modere |
| 3 | Multi-agents | Faible |
| 4 | Agent autonome long terme | Minimal |


In [ ]:
import json
import re
from typing import Any

# ============================================================
# AGENT LOOP MINIMAL -- from scratch
# ============================================================
class SimpleTool:
    """Representation d un outil callable par un agent"""
    def __init__(self, name, description, func, params_schema):
        self.name        = name
        self.description = description
        self.func        = func
        self.schema      = params_schema

    def run(self, **kwargs):
        return self.func(**kwargs)

    def to_dict(self):
        return {
            "name": self.name,
            "description": self.description,
            "parameters": self.schema
        }

# Outils pour un agent bancaire
def get_account_balance(account_id: str) -> dict:
    """Simule la recuperation du solde d un compte"""
    balances = {"ACC001": 15420.50, "ACC002": 3210.00, "ACC003": 89750.25}
    balance = balances.get(account_id, None)
    if balance is None:
        return {"error": f"Compte {account_id} introuvable"}
    return {"account_id": account_id, "balance": balance, "currency": "EUR"}

def transfer_funds(from_account: str, to_account: str, amount: float) -> dict:
    """Simule un virement bancaire"""
    if amount <= 0:
        return {"error": "Montant invalide"}
    if amount > 10000:
        return {"error": "Montant depasse la limite de virement en ligne (10000 EUR)"}
    return {
        "status": "success",
        "transaction_id": f"TXN{abs(hash(from_account+to_account))%100000:05d}",
        "from": from_account, "to": to_account, "amount": amount
    }

def search_transactions(account_id: str, limit: int = 5) -> list:
    """Simule la recherche de transactions"""
    import random; random.seed(42)
    ops = ["Achat en ligne", "Virement recu", "Prelevement EDF", "Retrait DAB", "Achat supermarche"]
    return [
        {"id": f"T{i:03d}", "label": random.choice(ops),
         "amount": round(random.uniform(-500, 2000), 2), "date": f"2024-03-{i+1:02d}"}
        for i in range(limit)
    ]

tools = [
    SimpleTool("get_account_balance", "Retourne le solde d un compte bancaire",
               get_account_balance, {"account_id": "string"}),
    SimpleTool("transfer_funds", "Effectue un virement entre deux comptes",
               transfer_funds, {"from_account": "string", "to_account": "string", "amount": "number"}),
    SimpleTool("search_transactions", "Recherche les dernieres transactions d un compte",
               search_transactions, {"account_id": "string", "limit": "integer"}),
]

print("=== Outils disponibles ===")
for tool in tools:
    print(f"  {tool.name:25s}: {tool.description}")

print("\n=== Test des outils ===")
print(f"  Balance ACC001: {get_account_balance('ACC001')}")
print(f"  Transfer:       {transfer_funds('ACC001', 'ACC002', 500.0)}")
print(f"  Transactions:   {search_transactions('ACC001', 3)}")


---
## 2. Patterns d’agence <a name='patterns'></a>

### ReAct (Reasoning + Acting)
Pattern le plus utilise, alterne entre reflexion et action :
```
Thought: Je dois connaitre le solde avant de faire le virement
Action: get_account_balance({"account_id": "ACC001"})
Observation: {"balance": 15420.50, "currency": "EUR"}
Thought: Le solde est suffisant. Je peux effectuer le virement.
Action: transfer_funds({"from_account": "ACC001", ...})
Observation: {"status": "success", ...}
Final Answer: Le virement de 500 EUR a ete effectue avec succes.
```

### Plan-and-Execute
1. **Planner** : decompose la tache en sous-taches
2. **Executor** : execute chaque sous-tache en ordre
3. **Replanner** : ajuste le plan selon les resultats

### Reflexion
L'agent critique ses propres sorties et se corrige :
```
Generation -> Critique (l agent evalue sa propre reponse) -> Revision
```

### CoT + Self-Consistency
- **Chain of Thought** : forcer une reflexion pas a pas
- **Self-Consistency** : echantillonner plusieurs CoT, voter sur la reponse finale

### Comparaison
| Pattern | Avantage | Inconvenient | Use case |
|---|---|---|---|
| ReAct | Simple, rapide | Pas de planification globale | Taches courtes |
| Plan-and-Execute | Taches complexes | Plus de tokens, plus lent | Workflows longs |
| Reflexion | Meilleure qualite | 2x plus de LLM calls | Generation de code |
| Self-Consistency | Robuste | N fois plus couteux | QA critique |


In [ ]:
import json

# ============================================================
# ReAct AGENT -- simulation
# ============================================================
class ReActAgent:
    """
    Agent ReAct (Reasoning + Acting)
    Boucle: Thought -> Action -> Observation -> (repeat) -> Final Answer
    """
    def __init__(self, tools, max_steps=5):
        self.tools    = {t.name: t for t in tools}
        self.max_steps = max_steps

    def _format_tools(self):
        return "
".join([
            f"- {name}: {t.description}  params={t.schema}"
            for name, t in self.tools.items()
        ])

    def _parse_action(self, text):
        """Extraire l action et ses parametres du texte LLM"""
        action_match = re.search(r"Action:\s*(\w+)\(({.*?})\)", text, re.DOTALL)
        if action_match:
            tool_name  = action_match.group(1)
            params_str = action_match.group(2)
            try:
                params = json.loads(params_str)
                return tool_name, params
            except:
                return None, None
        return None, None

    def _fake_llm(self, prompt, step, query):
        """Simule les reponses du LLM pour la demo"""
        if step == 0:
            return f"""Thought: Je dois d abord verifier le solde du compte ACC001 avant toute action.
Action: get_account_balance({{"account_id": "ACC001"}})"""
        elif step == 1:
            return f"""Thought: Le solde est suffisant pour le virement de 500 EUR. Je procede.
Action: transfer_funds({{"from_account": "ACC001", "to_account": "ACC002", "amount": 500.0}})"""
        elif step == 2:
            return f"""Thought: Le virement a reussi. Je vais aussi verifier les dernieres transactions.
Action: search_transactions({{"account_id": "ACC001", "limit": 3}})"""
        else:
            return """Final Answer: Le virement de 500 EUR du compte ACC001 vers ACC002 a ete effectue avec succes (TXN reference donnee). Le solde disponible avant virement etait de 15420.50 EUR."""

    def run(self, query):
        print(f"=== ReAct Agent ===")
        print(f"Query: {query}
")
        history = []

        for step in range(self.max_steps):
            # Appel LLM (simule)
            llm_output = self._fake_llm(None, step, query)
            print(f"--- Step {step+1} ---")
            print(llm_output)

            # Verifier si c est une reponse finale
            if "Final Answer:" in llm_output:
                final = llm_output.split("Final Answer:")[-1].strip()
                print(f"
=== Reponse finale ===
{final}")
                return final

            # Parser l action
            tool_name, params = self._parse_action(llm_output)
            if tool_name and tool_name in self.tools:
                observation = self.tools[tool_name].run(**params)
                print(f"Observation: {json.dumps(observation, ensure_ascii=False)}
")
                history.append({"thought": llm_output, "observation": observation})
            else:
                print("Observation: Aucune action valide detectee
")

        return "Max steps reached"

agent = ReActAgent(tools, max_steps=5)
agent.run("Effectue un virement de 500 EUR du compte ACC001 vers ACC002 et montre les transactions recentes")


In [ ]:
import json
from typing import List, Dict

# ============================================================
# PLAN-AND-EXECUTE -- from scratch
# ============================================================
class PlanAndExecuteAgent:
    """
    Plan-and-Execute:
    1. Planner decoupe la tache en etapes
    2. Executor execute chaque etape
    3. Replanner ajuste si necessaire
    """
    def __init__(self, tools):
        self.tools = {t.name: t for t in tools}

    def plan(self, task: str) -> List[Dict]:
        """Simule le planner LLM : decompose en sous-taches"""
        # En pratique: LLM genere ce plan
        return [
            {"step": 1, "tool": "get_account_balance", "params": {"account_id": "ACC001"}, "reason": "Verifier le solde disponible"},
            {"step": 2, "tool": "get_account_balance", "params": {"account_id": "ACC002"}, "reason": "Verifier le compte destinataire"},
            {"step": 3, "tool": "transfer_funds",       "params": {"from_account": "ACC001", "to_account": "ACC002", "amount": 1000.0}, "reason": "Effectuer le virement"},
            {"step": 4, "tool": "search_transactions",  "params": {"account_id": "ACC001", "limit": 5}, "reason": "Confirmer la transaction"},
        ]

    def execute_step(self, step: Dict, context: Dict) -> Dict:
        """Execute une etape du plan"""
        tool_name = step["tool"]
        params    = step["params"]
        # Resoudre les references au contexte si necessaire
        result = self.tools[tool_name].run(**params)
        return {"step": step["step"], "tool": tool_name, "result": result}

    def should_replan(self, step_result: Dict) -> bool:
        """Decide si le plan doit etre ajuste"""
        result = step_result.get("result", {})
        return "error" in result

    def run(self, task: str):
        print(f"=== Plan-and-Execute Agent ===")
        print(f"Tache: {task}
")

        plan = self.plan(task)
        print("Plan initial:")
        for step in plan:
            print(f"  Etape {step['step']}: {step['tool']}({step['params']})  => {step['reason']}")
        print()

        context = {}
        for step in plan:
            print(f"Execution etape {step['step']} : {step['tool']}")
            result = self.execute_step(step, context)
            print(f"  Resultat: {json.dumps(result['result'], ensure_ascii=False)}")
            context[f"step_{step['step']}"] = result["result"]

            if self.should_replan(result):
                print(f"  => ERREUR detectee, arret du plan")
                break
            print()

        print("=== Synthese ===")
        balance_before = context.get("step_1", {}).get("balance", "?")
        transfer_status = context.get("step_3", {}).get("status", "?")
        print(f"  Solde initial ACC001 : {balance_before} EUR")
        print(f"  Statut du virement   : {transfer_status}")
        print(f"  Transactions post-virement : {len(context.get('step_4', []))} enregistrees")

agent_pe = PlanAndExecuteAgent(tools)
agent_pe.run("Virer 1000 EUR de ACC001 vers ACC002 et confirmer l execution")


---
## 3. Tool Calling & Function Calling <a name='tools'></a>

### Principe
Les LLMs modernes (GPT-4, Claude, LLaMA) peuvent generer des appels de fonction structures en JSON :
```json
{
  "tool_calls": [{
    "id": "call_abc123",
    "type": "function",
    "function": {
      "name": "get_account_balance",
      "arguments": "{"account_id": "ACC001"}"
    }
  }]
}
```

### Format OpenAI Function Calling
```python
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current weather",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City name"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
            },
            "required": ["location"]
        }
    }
}]
```

### Parallel Tool Calling
Les LLMs peuvent appeler **plusieurs outils en parallele** dans une seule reponse :
```json
[
  {"name": "get_balance", "arguments": {"account_id": "ACC001"}},
  {"name": "get_balance", "arguments": {"account_id": "ACC002"}}
]
```

### Gestion des erreurs d’outils
- L'outil peut retourner une erreur
- L'agent doit pouvoir gerer ces erreurs (retry, fallback, escalade)
- Timeout : limiter le temps d'execution d'un outil


In [ ]:
import json
import time
from typing import List, Dict, Optional, Callable

# ============================================================
# TOOL REGISTRY -- gestion formelle des outils
# ============================================================
class ToolRegistry:
    """
    Registre central de tous les outils disponibles
    - Schema JSON Schema compatible (OpenAI format)
    - Validation des parametres
    - Execution avec timeout et gestion d erreurs
    """
    def __init__(self):
        self._tools: Dict[str, dict] = {}
        self._funcs: Dict[str, Callable] = {}

    def register(self, name: str, description: str, params_schema: dict, func: Callable):
        """Enregistrer un outil"""
        self._tools[name] = {
            "type": "function",
            "function": {
                "name": name,
                "description": description,
                "parameters": {
                    "type": "object",
                    "properties": params_schema,
                    "required": list(params_schema.keys())
                }
            }
        }
        self._funcs[name] = func
        return self

    def get_schemas(self) -> List[dict]:
        """Retourner les schemas pour le LLM"""
        return list(self._tools.values())

    def execute(self, tool_call: dict, timeout: float = 5.0) -> dict:
        """Executer un appel d outil avec gestion d erreurs"""
        name = tool_call.get("name") or tool_call.get("function", {}).get("name")
        args_raw = tool_call.get("arguments") or tool_call.get("function", {}).get("arguments", "{}")

        if name not in self._funcs:
            return {"error": f"Outil inconnu: {name}"}

        try:
            args = json.loads(args_raw) if isinstance(args_raw, str) else args_raw
            start = time.time()
            result = self._funcs[name](**args)
            elapsed = time.time() - start
            if elapsed > timeout:
                return {"error": f"Timeout apres {elapsed:.2f}s"}
            return {"result": result, "elapsed_ms": round(elapsed * 1000, 2)}
        except Exception as e:
            return {"error": str(e)}

    def execute_parallel(self, tool_calls: List[dict]) -> List[dict]:
        """Executer plusieurs outils (simule le parallelisme)"""
        return [self.execute(tc) for tc in tool_calls]

# Construction du registry
registry = ToolRegistry()
registry     .register("get_account_balance", "Retourne le solde d un compte",
              {"account_id": {"type": "string", "description": "Identifiant du compte"}},
              get_account_balance)     .register("transfer_funds", "Effectue un virement bancaire",
              {"from_account": {"type": "string"}, "to_account": {"type": "string"},
               "amount": {"type": "number", "description": "Montant en EUR"}},
              transfer_funds)     .register("search_transactions", "Recherche les transactions recentes",
              {"account_id": {"type": "string"}, "limit": {"type": "integer"}},
              search_transactions)

print("=== Tool Registry ===")
schemas = registry.get_schemas()
print(f"Outils enregistres: {len(schemas)}")
for s in schemas:
    print(f"  {s['function']['name']:25s}: {s['function']['description']}")

print("\n=== Execution d outils (format OpenAI) ===")
tool_calls = [
    {"name": "get_account_balance", "arguments": json.dumps({"account_id": "ACC001"})},
    {"name": "get_account_balance", "arguments": json.dumps({"account_id": "ACC002"})},
]
print("Parallel tool calls:")
results = registry.execute_parallel(tool_calls)
for tc, res in zip(tool_calls, results):
    print(f"  {tc['name']}({tc['arguments']}) => {res}")

print("\n=== Gestion des erreurs ===")
error_cases = [
    {"name": "transfer_funds", "arguments": json.dumps({"from_account": "ACC001", "to_account": "ACC002", "amount": 50000.0})},
    {"name": "nonexistent_tool", "arguments": "{}"},
    {"name": "get_account_balance", "arguments": json.dumps({"account_id": "UNKNOWN"})},
]
for tc in error_cases:
    result = registry.execute(tc)
    print(f"  {tc['name']}: {result}")


---
## 4. MCP — Model Context Protocol <a name='mcp'></a>

### Qu'est-ce que MCP ?
**Model Context Protocol** (Anthropic, nov 2024) : protocole standardise pour connecter les LLMs a des sources de donnees et des outils externes.

> "MCP est le USB-C des connexions LLM-outil"

### Probleme resolu
Avant MCP : chaque integration LLM-outil etait ad hoc. N modeles * M outils = N*M integrations.
Avec MCP : N modeles + M outils = N+M implementations.

### Architecture
```
LLM Application (Host)
    |
    | MCP Protocol (JSON-RPC over stdio/SSE/HTTP)
    |
MCP Server
    |
    +-- Resources (fichiers, BDD, APIs)
    +-- Tools (fonctions callables)
    +-- Prompts (templates)
```

### Primitives MCP
| Primitive | Direction | Description |
|---|---|---|
| **Resources** | Server -> Client | Expose des donnees (fichiers, BDD, API responses) |
| **Tools** | Client -> Server | Fonctions callables par le LLM |
| **Prompts** | Server -> Client | Templates de prompts pre-definis |
| **Sampling** | Server -> Client | Demander au LLM de generer |

### Transports
- **stdio** : communication par stdin/stdout (processus local)
- **SSE (Server-Sent Events)** : HTTP streaming (remote)
- **HTTP+JSON-RPC** : requete/reponse classique

### Exemples de MCP servers
- `@anthropic/mcp-server-filesystem` : acces aux fichiers
- `@anthropic/mcp-server-github` : GitHub API
- `mcp-server-postgres` : base de donnees PostgreSQL
- `mcp-server-brave-search` : recherche web


In [ ]:
import json
import uuid

# ============================================================
# MCP PROTOCOL SIMULATION -- JSON-RPC 2.0
# ============================================================
class MCPMessage:
    """Message JSON-RPC 2.0 (base du protocole MCP)"""
    @staticmethod
    def request(method: str, params: dict = None, id: str = None) -> dict:
        return {
            "jsonrpc": "2.0",
            "id": id or str(uuid.uuid4())[:8],
            "method": method,
            "params": params or {}
        }

    @staticmethod
    def response(id: str, result: dict) -> dict:
        return {"jsonrpc": "2.0", "id": id, "result": result}

    @staticmethod
    def error(id: str, code: int, message: str) -> dict:
        return {"jsonrpc": "2.0", "id": id, "error": {"code": code, "message": message}}

    @staticmethod
    def notification(method: str, params: dict = None) -> dict:
        return {"jsonrpc": "2.0", "method": method, "params": params or {}}


class MCPServer:
    """
    Serveur MCP simplifie :
    - Gere les primitives Resources, Tools, Prompts
    - Repond aux requetes JSON-RPC
    """
    def __init__(self, name: str, version: str = "1.0.0"):
        self.name    = name
        self.version = version
        self._tools     = {}
        self._resources = {}
        self._prompts   = {}

    def register_tool(self, name: str, description: str, schema: dict, func):
        self._tools[name] = {"description": description, "schema": schema, "func": func}

    def register_resource(self, uri: str, name: str, content: str, mime: str = "text/plain"):
        self._resources[uri] = {"name": name, "content": content, "mimeType": mime}

    def register_prompt(self, name: str, description: str, template: str):
        self._prompts[name] = {"description": description, "template": template}

    def handle(self, message: dict) -> dict:
        """Dispatcher principal des messages JSON-RPC"""
        method = message.get("method")
        params = message.get("params", {})
        msg_id = message.get("id")

        handlers = {
            "initialize":      self._handle_initialize,
            "tools/list":      self._handle_tools_list,
            "tools/call":      self._handle_tools_call,
            "resources/list":  self._handle_resources_list,
            "resources/read":  self._handle_resources_read,
            "prompts/list":    self._handle_prompts_list,
            "prompts/get":     self._handle_prompts_get,
        }

        handler = handlers.get(method)
        if handler is None:
            return MCPMessage.error(msg_id, -32601, f"Method not found: {method}")
        try:
            result = handler(params)
            return MCPMessage.response(msg_id, result)
        except Exception as e:
            return MCPMessage.error(msg_id, -32603, str(e))

    def _handle_initialize(self, params):
        return {
            "protocolVersion": "2024-11-05",
            "serverInfo": {"name": self.name, "version": self.version},
            "capabilities": {"tools": {}, "resources": {}, "prompts": {}}
        }

    def _handle_tools_list(self, params):
        return {"tools": [
            {"name": n, "description": t["description"],
             "inputSchema": {"type": "object", "properties": t["schema"]}}
            for n, t in self._tools.items()
        ]}

    def _handle_tools_call(self, params):
        name = params.get("name")
        args = params.get("arguments", {})
        if name not in self._tools:
            raise ValueError(f"Tool not found: {name}")
        result = self._tools[name]["func"](**args)
        return {"content": [{"type": "text", "text": json.dumps(result, ensure_ascii=False)}]}

    def _handle_resources_list(self, params):
        return {"resources": [
            {"uri": uri, "name": r["name"], "mimeType": r["mimeType"]}
            for uri, r in self._resources.items()
        ]}

    def _handle_resources_read(self, params):
        uri = params.get("uri")
        if uri not in self._resources:
            raise ValueError(f"Resource not found: {uri}")
        r = self._resources[uri]
        return {"contents": [{"uri": uri, "mimeType": r["mimeType"], "text": r["content"]}]}

    def _handle_prompts_list(self, params):
        return {"prompts": [
            {"name": n, "description": p["description"]}
            for n, p in self._prompts.items()
        ]}

    def _handle_prompts_get(self, params):
        name = params.get("name")
        args = params.get("arguments", {})
        if name not in self._prompts:
            raise ValueError(f"Prompt not found: {name}")
        tmpl = self._prompts[name]["template"]
        for k, v in args.items():
            tmpl = tmpl.replace("{" + k + "}", str(v))
        return {"messages": [{"role": "user", "content": {"type": "text", "text": tmpl}}]}

# ============================================================
# MCP SERVER BANCAIRE
# ============================================================
banking_server = MCPServer("bnp-banking-mcp", "1.0.0")

# Outils
banking_server.register_tool("get_balance", "Retourne le solde d un compte",
    {"account_id": {"type": "string"}}, get_account_balance)
banking_server.register_tool("transfer", "Effectue un virement",
    {"from_account": {"type": "string"}, "to_account": {"type": "string"}, "amount": {"type": "number"}},
    transfer_funds)

# Ressources
banking_server.register_resource(
    "bnp://docs/api-reference", "API Reference",
    "# BNP Banking API
## Endpoints
GET /balance/{id}
POST /transfer
...", "text/markdown")
banking_server.register_resource(
    "bnp://data/exchange-rates", "Exchange Rates",
    json.dumps({"EUR/USD": 1.085, "EUR/GBP": 0.856, "EUR/JPY": 161.2}), "application/json")

# Prompts
banking_server.register_prompt("analyze_account", "Analyse un compte bancaire",
    "Analyse le compte {account_id}. Evalue la sante financiere et propose des recommandations.")

print("=== MCP Server BNP Banking ===")

# 1. Initialize
req = MCPMessage.request("initialize", {"clientInfo": {"name": "claude", "version": "3.5"}})
resp = banking_server.handle(req)
print(f"1. Initialize: {resp['result']['serverInfo']}")

# 2. List tools
req = MCPMessage.request("tools/list")
resp = banking_server.handle(req)
print(f"\n2. Tools ({len(resp['result']['tools'])} disponibles):")
for t in resp["result"]["tools"]:
    print(f"   - {t['name']}: {t['description']}")

# 3. Call tool
req = MCPMessage.request("tools/call", {"name": "get_balance", "arguments": {"account_id": "ACC001"}})
resp = banking_server.handle(req)
print(f"\n3. Tool call (get_balance ACC001): {resp['result']['content'][0]['text']}")

# 4. List resources
req = MCPMessage.request("resources/list")
resp = banking_server.handle(req)
print(f"\n4. Resources ({len(resp['result']['resources'])} disponibles):")
for r in resp["result"]["resources"]:
    print(f"   - {r['uri']}: {r['name']}")

# 5. Read resource
req = MCPMessage.request("resources/read", {"uri": "bnp://data/exchange-rates"})
resp = banking_server.handle(req)
print(f"\n5. Resource read (exchange-rates): {resp['result']['contents'][0]['text']}")

# 6. Get prompt
req = MCPMessage.request("prompts/get", {"name": "analyze_account", "arguments": {"account_id": "ACC001"}})
resp = banking_server.handle(req)
print(f"\n6. Prompt: {resp['result']['messages'][0]['content']['text']}")


---
## 5. A2A — Agent-to-Agent Protocol <a name='a2a'></a>

### Qu'est-ce que A2A ?
**Agent-to-Agent Protocol** (Google, avril 2025) : protocole standardise pour la communication entre agents IA de differents fournisseurs.

### Probleme resolu
Sans A2A : les agents d'ecosystemes differents ne peuvent pas collaborer directement.
Avec A2A : un agent Claude peut deleguer a un agent GPT-4, qui peut deleguer a un agent Gemini.

### Concepts cles

**Agent Card** : description publique des capacites d'un agent (comme un manifeste)
```json
{
  "name": "BNP Fraud Detection Agent",
  "description": "Detecte les transactions frauduleuses",
  "url": "https://agents.bnp.com/fraud-detector",
  "capabilities": ["fraud_analysis", "risk_scoring"],
  "skills": [{"name": "analyze_transaction", "description": "..."}]
}
```

**Task** : unite de travail envoyee d'un agent a un autre
```json
{
  "id": "task_123",
  "sessionId": "session_abc",
  "message": {"role": "user", "content": "Analyse cette transaction: ..."},
  "status": "submitted"
}
```

**Task Status** : `submitted` -> `working` -> `completed` | `failed` | `input-required`

### Flux A2A
```
Agent A (orchestrateur)
  |
  | POST /tasks/send
  v
Agent B (specialiste)
  |
  | Stream d evenements (SSE)
  | task/status-update
  | task/artifact-update
  |
  v
Agent A recoit le resultat


In [ ]:
import json
import uuid
from enum import Enum
from typing import Optional

# ============================================================
# A2A PROTOCOL -- simulation
# ============================================================
class TaskStatus(str, Enum):
    SUBMITTED      = "submitted"
    WORKING        = "working"
    COMPLETED      = "completed"
    FAILED         = "failed"
    INPUT_REQUIRED = "input-required"
    CANCELLED      = "cancelled"

class A2ATask:
    """Representation d une tache A2A"""
    def __init__(self, message: str, session_id: str = None):
        self.id         = str(uuid.uuid4())[:8]
        self.session_id = session_id or str(uuid.uuid4())[:8]
        self.message    = {"role": "user", "content": [{"type": "text", "text": message}]}
        self.status     = TaskStatus.SUBMITTED
        self.artifacts  = []
        self.history    = [{"status": self.status, "message": "Task created"}]

    def update_status(self, status: TaskStatus, message: str = ""):
        self.status = status
        self.history.append({"status": status, "message": message})

    def add_artifact(self, name: str, content: str, mime: str = "text/plain"):
        self.artifacts.append({"name": name, "content": content, "mimeType": mime})

    def to_dict(self):
        return {
            "id": self.id,
            "sessionId": self.session_id,
            "status": {"state": self.status},
            "message": self.message,
            "artifacts": self.artifacts
        }

class A2AAgent:
    """
    Agent A2A generique:
    - Expose une Agent Card
    - Recoit et traite des taches
    - Peut deleguer a d autres agents
    """
    def __init__(self, name: str, description: str, skills: list):
        self.name        = name
        self.description = description
        self.skills      = skills
        self.tasks: dict = {}

    @property
    def agent_card(self) -> dict:
        return {
            "name": self.name,
            "description": self.description,
            "url": f"https://agents.example.com/{self.name.lower().replace(' ', '-')}",
            "version": "1.0.0",
            "capabilities": {"streaming": True, "pushNotifications": False},
            "skills": [{"id": s["id"], "name": s["name"], "description": s["description"]}
                       for s in self.skills]
        }

    def send_task(self, message: str) -> A2ATask:
        """Recevoir une tache et commencer le traitement"""
        task = A2ATask(message)
        self.tasks[task.id] = task
        task.update_status(TaskStatus.WORKING, f"Agent {self.name} traite la tache")
        # Traitement (simule)
        self._process(task)
        return task

    def _process(self, task: A2ATask):
        """Traitement specifique a chaque agent (override)"""
        task.update_status(TaskStatus.COMPLETED, "Traitement termine")

    def get_task(self, task_id: str) -> Optional[A2ATask]:
        return self.tasks.get(task_id)

# Agents specialises BNP
class FraudDetectionAgent(A2AAgent):
    def __init__(self):
        super().__init__(
            "BNP Fraud Detection Agent",
            "Detecte et analyse les transactions potentiellement frauduleuses",
            [{"id": "fraud_analysis", "name": "Fraud Analysis",
              "description": "Analyse une transaction et retourne un score de risque"}]
        )

    def _process(self, task: A2ATask):
        task.update_status(TaskStatus.WORKING, "Analyse des patterns de fraude en cours...")
        import random; random.seed(42)
        score = round(random.uniform(0, 1), 3)
        risk  = "ELEVE" if score > 0.7 else "MOYEN" if score > 0.4 else "FAIBLE"
        result = {
            "fraud_score": score, "risk_level": risk,
            "flags": ["montant inhabituel", "pays inhabituel"] if score > 0.7 else [],
            "recommendation": "Bloquer" if score > 0.7 else "Surveiller" if score > 0.4 else "Approuver"
        }
        task.add_artifact("fraud_report", json.dumps(result, ensure_ascii=False), "application/json")
        task.update_status(TaskStatus.COMPLETED, f"Score de fraude: {score} ({risk})")

class CreditScoringAgent(A2AAgent):
    def __init__(self):
        super().__init__(
            "BNP Credit Scoring Agent",
            "Calcule le score de credit d un client",
            [{"id": "credit_score", "name": "Credit Score",
              "description": "Calcule le score de credit et la capacite d emprunt"}]
        )

    def _process(self, task: A2ATask):
        task.update_status(TaskStatus.WORKING, "Calcul du score de credit...")
        import random; random.seed(123)
        score = random.randint(600, 850)
        result = {
            "credit_score": score,
            "rating": "Excellent" if score > 750 else "Bon" if score > 680 else "Moyen",
            "max_loan": round(score * 200, -2),
            "interest_rate": round(5 - (score - 600) / 250, 2)
        }
        task.add_artifact("credit_report", json.dumps(result, ensure_ascii=False), "application/json")
        task.update_status(TaskStatus.COMPLETED, f"Score: {score}")

print("=== A2A Protocol ===")
fraud_agent  = FraudDetectionAgent()
credit_agent = CreditScoringAgent()

print("\n--- Agent Cards ---")
for agent in [fraud_agent, credit_agent]:
    card = agent.agent_card
    print(f"\n{card['name']}:")
    print(f"  URL:    {card['url']}")
    print(f"  Skills: {[s['name'] for s in card['skills']]}")

print("\n--- Execution de taches A2A ---")
task1 = fraud_agent.send_task("Analyse la transaction: virement de 9800 EUR vers un compte en Lituanie")
print(f"\nTask fraud {task1.id} (status: {task1.status}):")
if task1.artifacts:
    print(f"  Artifact: {task1.artifacts[0]['content']}")
print(f"  History: {[h['status'] for h in task1.history]}")

task2 = credit_agent.send_task("Calcule le score de credit pour le client ID 12345")
print(f"\nTask credit {task2.id} (status: {task2.status}):")
if task2.artifacts:
    print(f"  Artifact: {task2.artifacts[0]['content']}")


---
## 6. AG-UI — Agent-User Interaction Protocol <a name='agui'></a>

### Qu'est-ce que AG-UI ?
**AG-UI** (CopilotKit, 2025) : protocole leger de streaming entre agents IA et interfaces utilisateur.

### Probleme resolu
Rendre l'execution d'agents **transparente et interactive** pour l'utilisateur :
- Voir l'agent "penser" en temps reel
- Intervenir pendant l'execution (human-in-the-loop)
- Voir l'utilisation des outils en direct

### Evenements AG-UI (Server-Sent Events)
```
RUN_STARTED        : L agent commence a traiter
TEXT_MESSAGE_START : Debut d un message texte
TEXT_MESSAGE_DELTA : Token en streaming
TEXT_MESSAGE_END   : Fin du message
TOOL_CALL_START    : L agent appelle un outil
TOOL_CALL_END      : Resultat de l outil recu
STATE_SNAPSHOT     : Etat partage agent <-> UI
STATE_DELTA        : Mise a jour incrementale d etat
MESSAGES_SNAPSHOT  : Historique complet
RUN_FINISHED       : L agent a termine
RUN_ERROR          : Erreur d execution
```

### Interactions bidirectionnelles
```
UI -> Agent : nouveau message utilisateur
UI -> Agent : interruption (stop)
UI -> Agent : mise a jour d etat (ex: formulaire rempli)
Agent -> UI : streaming de tokens
Agent -> UI : utilisation d outils (visible)
Agent -> UI : demande d input (human-in-the-loop)
```

### Comparaison avec d'autres protocols
| | MCP | A2A | AG-UI |
|---|---|---|---|
| Cible | LLM <-> outils | Agent <-> Agent | Agent <-> UI |
| Direction | Bidirectionnel | Bidirectionnel | Bidirectionnel |
| Transport | stdio/SSE | HTTP+SSE | SSE/WebSocket |
| Standard | Anthropic | Google | CopilotKit |
| Use case | Plugins LLM | Multi-agent | Frontend agent |


In [ ]:
import json
import time
from enum import Enum
from typing import Generator

# ============================================================
# AG-UI EVENTS -- simulation du streaming
# ============================================================
class AGUIEventType(str, Enum):
    RUN_STARTED          = "RUN_STARTED"
    TEXT_MESSAGE_START   = "TEXT_MESSAGE_START"
    TEXT_MESSAGE_DELTA   = "TEXT_MESSAGE_DELTA"
    TEXT_MESSAGE_END     = "TEXT_MESSAGE_END"
    TOOL_CALL_START      = "TOOL_CALL_START"
    TOOL_CALL_END        = "TOOL_CALL_END"
    STATE_SNAPSHOT       = "STATE_SNAPSHOT"
    STATE_DELTA          = "STATE_DELTA"
    RUN_FINISHED         = "RUN_FINISHED"
    RUN_ERROR            = "RUN_ERROR"

def make_event(event_type: AGUIEventType, data: dict) -> dict:
    """Creer un evenement AG-UI"""
    return {"type": event_type, "timestamp": time.time(), **data}

class AGUIAgent:
    """
    Agent qui streame ses evenements au format AG-UI
    Compatible avec CopilotKit, CoAgent, etc.
    """
    def __init__(self, tools_registry):
        self.registry = tools_registry
        self.state = {}   # etat partage avec l UI

    def stream(self, user_message: str) -> Generator[dict, None, None]:
        """
        Genere un flux d evenements AG-UI
        Simule la boucle ReAct avec streaming
        """
        run_id = str(uuid.uuid4())[:8]
        msg_id = str(uuid.uuid4())[:8]

        # 1. Debut d execution
        yield make_event(AGUIEventType.RUN_STARTED, {"runId": run_id})
        yield make_event(AGUIEventType.STATE_SNAPSHOT, {
            "snapshot": {"status": "thinking", "tool_calls": [], "message_count": 1}
        })

        # 2. Streaming de la reflexion de l agent
        yield make_event(AGUIEventType.TEXT_MESSAGE_START, {"messageId": msg_id, "role": "assistant"})
        thought = "Je vais analyser votre demande et consulter les outils disponibles..."
        for word in thought.split():
            yield make_event(AGUIEventType.TEXT_MESSAGE_DELTA, {"messageId": msg_id, "delta": word + " "})
        yield make_event(AGUIEventType.TEXT_MESSAGE_END, {"messageId": msg_id})

        # 3. Appel d outil
        tool_call_id = str(uuid.uuid4())[:8]
        yield make_event(AGUIEventType.TOOL_CALL_START, {
            "toolCallId": tool_call_id,
            "toolCallName": "get_account_balance",
            "parentMessageId": msg_id
        })
        # Simulation du traitement
        tool_result = get_account_balance("ACC001")
        yield make_event(AGUIEventType.TOOL_CALL_END, {
            "toolCallId": tool_call_id,
            "result": json.dumps(tool_result, ensure_ascii=False)
        })

        # 4. Mise a jour d etat
        self.state["last_balance"] = tool_result.get("balance")
        yield make_event(AGUIEventType.STATE_DELTA, {
            "delta": [{"op": "add", "path": "/last_balance", "value": tool_result.get("balance")}]
        })

        # 5. Reponse finale
        final_msg_id = str(uuid.uuid4())[:8]
        yield make_event(AGUIEventType.TEXT_MESSAGE_START, {"messageId": final_msg_id, "role": "assistant"})
        response_text = f"Le solde du compte ACC001 est de {tool_result.get('balance', '?')} EUR. Comment puis-je vous aider davantage ?"
        for word in response_text.split():
            yield make_event(AGUIEventType.TEXT_MESSAGE_DELTA, {"messageId": final_msg_id, "delta": word + " "})
        yield make_event(AGUIEventType.TEXT_MESSAGE_END, {"messageId": final_msg_id})

        # 6. Fin d execution
        yield make_event(AGUIEventType.RUN_FINISHED, {
            "runId": run_id,
            "result": {"status": "success", "message": response_text}
        })

# Demo du streaming AG-UI
print("=== AG-UI Streaming Events ===")
print("(Simule ce qu une interface React/Vue verrait en Server-Sent Events)
")

ag_agent = AGUIAgent(registry)
events   = list(ag_agent.stream("Quel est le solde de mon compte ACC001 ?"))

for event in events:
    event_type = event["type"]
    if event_type == AGUIEventType.TEXT_MESSAGE_DELTA:
        print(f"  [STREAM] {event['delta']}", end="", flush=True)
    else:
        display = {k: v for k, v in event.items() if k not in ["timestamp", "type"]}
        print(f"
  [{event_type}] {json.dumps(display, ensure_ascii=False)[:80]}")

print(f"

Etat final de l agent: {ag_agent.state}")
print(f"Total evenements streames: {len(events)}")


---
## 7. Orchestration multi-agents <a name='orchestration'></a>

### Patterns d'orchestration

**Supervisor (Hub and Spoke)**
```
Orchestrateur central
    |---- Agent Specialiste 1
    |---- Agent Specialiste 2
    |---- Agent Specialiste 3
```
- L'orchestrateur decide qui appelle
- Simple a debugger
- Goulot d'etranglement si l'orchestrateur tombe

**Pipeline (Chaine)**
```
Agent A -> Agent B -> Agent C -> Resultat
```
- Chaque agent transforme et passe au suivant
- Simple, predictable
- Pas de branchements

**Swarm (essaim)**
- Agents pairs, se passent le "controle"
- Dynamique, resilient
- Plus complexe a debugger

**Hierarchique**
```
Manager Agent
    |---- Team Lead A
    |         |---- Worker 1
    |         |---- Worker 2
    |---- Team Lead B
              |---- Worker 3
```

### Frameworks populaires
| Framework | Pattern | Langage | Notes |
|---|---|---|---|
| LangGraph | Graph (Supervisor/Swarm) | Python | LangChain ecosystem |
| CrewAI | Hierarchique | Python | Multi-role, task-based |
| AutoGen | Conversation | Python | Microsoft, code focus |
| OpenAI Swarm | Swarm | Python | Educatif, leger |
| Semantic Kernel | Pipeline | Python/C# | Microsoft, enterprise |


In [ ]:
import json
from typing import Dict, List, Optional, Callable

# ============================================================
# ORCHESTRATEUR MULTI-AGENTS -- pattern Supervisor
# ============================================================
class AgentMessage:
    def __init__(self, role: str, content: str, agent_name: str = None):
        self.role       = role
        self.content    = content
        self.agent_name = agent_name

    def to_dict(self):
        d = {"role": self.role, "content": self.content}
        if self.agent_name:
            d["name"] = self.agent_name
        return d

class BaseAgent:
    def __init__(self, name: str, description: str, system_prompt: str):
        self.name          = name
        self.description   = description
        self.system_prompt = system_prompt
        self.memory: List[AgentMessage] = []

    def run(self, task: str, context: dict = None) -> str:
        """Override dans les sous-classes"""
        raise NotImplementedError

class SupervisorOrchestrator:
    """
    Orchestrateur Supervisor :
    - Decide quel agent specialiste appeler selon la tache
    - Combine les resultats
    - Peut appeler plusieurs agents en sequence ou en parallele
    """
    def __init__(self, agents: List[BaseAgent]):
        self.agents = {a.name: a for a in agents}
        self.execution_log: List[dict] = []

    def _route(self, task: str) -> List[str]:
        """Router la tache vers le(s) bon(s) agent(s) (simule LLM routing)"""
        task_lower = task.lower()
        selected = []
        routing_rules = [
            (["fraude", "suspicious", "risque", "transaction suspecte"], "FraudAgent"),
            (["credit", "emprunt", "pret", "score"],                     "CreditAgent"),
            (["compte", "solde", "balance", "virement", "transfer"],     "BankingAgent"),
            (["rapport", "analyse", "report", "synthese"],               "AnalysisAgent"),
        ]
        for keywords, agent_name in routing_rules:
            if any(kw in task_lower for kw in keywords) and agent_name in self.agents:
                selected.append(agent_name)
        return selected or list(self.agents.keys())[:1]

    def run(self, task: str, parallel: bool = False) -> dict:
        """Executer la tache via le bon agent"""
        print(f"
=== Supervisor Orchestrator ===")
        print(f"Tache: {task}")

        agents_to_call = self._route(task)
        print(f"Agents selectionnes: {agents_to_call}")
        print()

        results = {}
        for agent_name in agents_to_call:
            agent = self.agents[agent_name]
            print(f"  --> Appel agent: {agent_name}")
            result = agent.run(task)
            results[agent_name] = result
            self.execution_log.append({
                "task": task, "agent": agent_name, "result": result
            })
            print(f"      Resultat: {result[:100]}...")

        # Synthese
        if len(results) > 1:
            synthesis = "Synthese multi-agents:
" + "
".join([
                f"- {name}: {res[:80]}" for name, res in results.items()
            ])
        else:
            synthesis = list(results.values())[0]

        print(f"
Synthese finale: {synthesis[:150]}")
        return {"agents_called": agents_to_call, "results": results, "synthesis": synthesis}

# Agents specialises
class FraudAgent(BaseAgent):
    def __init__(self):
        super().__init__("FraudAgent", "Agent de detection de fraude",
                         "Tu es un expert en detection de fraude bancaire.")
    def run(self, task, context=None):
        import random; random.seed(99)
        score = round(random.uniform(0.1, 0.9), 2)
        return f"Score de fraude: {score}/1.0. Niveau: {'ELEVE' if score>0.7 else 'MOYEN' if score>0.4 else 'FAIBLE'}. Recommandation: {'Bloquer' if score>0.7 else 'Surveiller'}."

class CreditAgent(BaseAgent):
    def __init__(self):
        super().__init__("CreditAgent", "Agent de scoring credit",
                         "Tu es un expert en evaluation du risque credit.")
    def run(self, task, context=None):
        import random; random.seed(55)
        score = random.randint(620, 820)
        return f"Score credit: {score}/850. Capacite d emprunt estimee: {score*150:,} EUR. Taux propose: {round(5-(score-600)/250,2)}%."

class BankingAgent(BaseAgent):
    def __init__(self):
        super().__init__("BankingAgent", "Agent operations bancaires",
                         "Tu geres les operations bancaires courantes.")
    def run(self, task, context=None):
        balance = get_account_balance("ACC001")
        return f"Operations bancaires: solde={balance['balance']} {balance['currency']}. Compte actif et en bonne sante."

class AnalysisAgent(BaseAgent):
    def __init__(self):
        super().__init__("AnalysisAgent", "Agent d analyse et synthese",
                         "Tu produis des analyses et syntheses financieres.")
    def run(self, task, context=None):
        return "Analyse complete: profil client stable, historique de paiement excellent, aucune alerte reglementaire detectee. Recommandation: client premium."

orchestrator = SupervisorOrchestrator([FraudAgent(), CreditAgent(), BankingAgent(), AnalysisAgent()])
r1 = orchestrator.run("Analyse cette transaction suspecte de 9500 EUR")
r2 = orchestrator.run("Quel est le score de credit et le solde de ce client ?")
print(f"
=== Log d execution ({len(orchestrator.execution_log)} appels) ===")
for log in orchestrator.execution_log:
    print(f"  [{log['agent']}] <- '{log['task'][:50]}...'")


---
## 8. Evaluation et fiabilité des agents <a name='eval'></a>

### Dimensions d’evaluation

**Correctness** : L’agent produit-il la bonne reponse ?
- Exact Match, F1, BLEU pour les sorties textuelles
- Unit tests pour les sorties structurees

**Faithfulness** : L’agent utilise-t-il correctement ses outils ?
- Les appels d’outils sont-ils valides ?
- Les resultats sont-ils correctement interpretes ?

**Efficiency** : L’agent est-il economique ?
- Nombre d’etapes pour resoudre la tache
- Tokens consommes (prompt + completion)
- Latence totale

**Robustness** : L’agent gere-t-il les cas limites ?
- Erreurs d’outils
- Informations manquantes ou contradictoires
- Boucles infinies

### Risques specifiques aux agents
| Risque | Description | Mitigation |
|---|---|---|
| Prompt injection | Texte malveillant dans l env qui detourne l agent | Sandboxing, validation |
| Tool misuse | Appel d outil avec mauvais params | Schema validation, dry-run |
| Infinite loop | L agent boucle indefiniment | Max steps, timeout |
| Data leakage | L agent exfilttre des donnees sensibles | Output filtering, RBAC |
| Hallucinated tools | L agent invente des outils inexistants | Tool registry strict |

### Benchmarks agents
- **GAIA** : taches du monde reel necessitant plusieurs outils
- **SWE-bench** : resolution de bugs GitHub avec acces au code
- **AgentBench** : environnements interactifs varies
- **HotPotQA** : QA multi-sauts necessitant du raisonnement


In [ ]:
import json
import time
from typing import List, Dict

# ============================================================
# EVALUATION FRAMEWORK POUR AGENTS
# ============================================================
class AgentEvaluator:
    """
    Framework d evaluation pour les agents :
    - Correctness (exact match ou similarity)
    - Tool usage (appels corrects, params valides)
    - Efficiency (nb d etapes, tokens)
    - Safety (pas de comportements dangereux)
    """
    def __init__(self):
        self.results: List[Dict] = []

    def evaluate(self, agent_run: dict, expected: dict) -> dict:
        """
        agent_run: {"answer": str, "steps": list, "tool_calls": list, "tokens": int}
        expected:  {"answer": str, "required_tools": list, "max_steps": int}
        """
        metrics = {}

        # 1. Correctness
        if "answer" in expected:
            overlap = self._token_overlap(agent_run.get("answer",""), expected["answer"])
            metrics["exact_match"] = int(agent_run.get("answer","").strip().lower() ==
                                         expected["answer"].strip().lower())
            metrics["f1_score"]    = overlap

        # 2. Tool usage
        if "required_tools" in expected:
            called_tools = [tc.get("name","") for tc in agent_run.get("tool_calls", [])]
            required     = set(expected["required_tools"])
            called_set   = set(called_tools)
            metrics["tool_recall"]    = len(required & called_set) / len(required) if required else 1.0
            metrics["tool_precision"] = len(required & called_set) / len(called_set) if called_set else 0.0
            metrics["unnecessary_tools"] = list(called_set - required)

        # 3. Efficiency
        n_steps = len(agent_run.get("steps", []))
        max_steps = expected.get("max_steps", float("inf"))
        metrics["n_steps"]    = n_steps
        metrics["n_tokens"]   = agent_run.get("tokens", 0)
        metrics["step_ratio"] = n_steps / max_steps if max_steps != float("inf") else None
        metrics["efficient"]  = n_steps <= max_steps

        # 4. Safety
        forbidden = ["DROP TABLE", "rm -rf", "os.system", "exec(", "eval("]
        all_text  = json.dumps(agent_run)
        metrics["safety_violations"] = [f for f in forbidden if f in all_text]
        metrics["safe"] = len(metrics["safety_violations"]) == 0

        # Score global
        scores = []
        if "f1_score" in metrics:    scores.append(metrics["f1_score"])
        if "tool_recall" in metrics: scores.append(metrics["tool_recall"])
        if "efficient" in metrics:   scores.append(float(metrics["efficient"]))
        if "safe" in metrics:        scores.append(float(metrics["safe"]))
        metrics["overall_score"] = sum(scores) / len(scores) if scores else 0.0

        self.results.append({"metrics": metrics})
        return metrics

    def _token_overlap(self, pred: str, ref: str) -> float:
        pred_tok = set(pred.lower().split())
        ref_tok  = set(ref.lower().split())
        if not ref_tok:
            return 0.0
        prec = len(pred_tok & ref_tok) / len(pred_tok) if pred_tok else 0
        rec  = len(pred_tok & ref_tok) / len(ref_tok)
        return 2 * prec * rec / (prec + rec + 1e-10)

    def summary(self) -> dict:
        if not self.results:
            return {}
        keys = self.results[0]["metrics"].keys()
        summary = {}
        for k in keys:
            vals = [r["metrics"][k] for r in self.results if isinstance(r["metrics"].get(k), (int, float))]
            if vals:
                import numpy as np
                summary[k] = {"mean": np.mean(vals), "std": np.std(vals)}
        return summary

# Tests d evaluation
evaluator = AgentEvaluator()
test_cases = [
    {
        "run": {
            "answer": "Le solde du compte ACC001 est de 15420.50 EUR",
            "steps": ["think", "call_get_balance", "answer"],
            "tool_calls": [{"name": "get_account_balance", "args": {"account_id": "ACC001"}}],
            "tokens": 450
        },
        "expected": {
            "answer": "Le solde est 15420.50 EUR",
            "required_tools": ["get_account_balance"],
            "max_steps": 3
        }
    },
    {
        "run": {
            "answer": "Virement effectue avec succes, reference TXN00345",
            "steps": ["think", "check_balance", "transfer", "confirm", "summarize"],
            "tool_calls": [
                {"name": "get_account_balance", "args": {}},
                {"name": "transfer_funds", "args": {}},
                {"name": "search_transactions", "args": {}}   # outil superflu
            ],
            "tokens": 1200
        },
        "expected": {
            "answer": "Virement de 500 EUR effectue",
            "required_tools": ["get_account_balance", "transfer_funds"],
            "max_steps": 4
        }
    },
]

print("=== Evaluation des agents ===")
print(f"{'':3s}  {'F1':>6}  {'Tool R.':>8}  {'Tool P.':>8}  {'Steps':>6}  {'Tokens':>7}  {'Safe':>6}  {'Overall':>8}")
for i, tc in enumerate(test_cases):
    m = evaluator.evaluate(tc["run"], tc["expected"])
    print(f"  {i+1}  {m.get('f1_score',0):>6.3f}  {m.get('tool_recall',0):>8.3f}  "
          f"{m.get('tool_precision',0):>8.3f}  {m.get('n_steps',0):>6}  "
          f"{m.get('n_tokens',0):>7}  {str(m.get('safe',True)):>6}  {m.get('overall_score',0):>8.3f}")
    if m.get("unnecessary_tools"):
        print(f"     Outils superflus: {m['unnecessary_tools']}")

print("\n=== Risques et mitigations ===")
risks = [
    ("Prompt injection", "Texte malveillant dans l env detourne l agent", "Sandboxing + validation input"),
    ("Infinite loop",    "L agent boucle indefiniment",                   "max_steps + timeout"),
    ("Tool misuse",      "Mauvais params passes a un outil",              "JSON Schema validation + dry-run"),
    ("Data leakage",     "Exfiltration de donnees sensibles",             "Output filtering + RBAC"),
    ("Reward hacking",   "Optimise la metrique, pas l objectif reel",     "Evaluation humaine + diverse metrics"),
]
for risk, desc, mitigation in risks:
    print(f"  [{risk}] {desc}")
    print(f"    => {mitigation}")


---
## 9. Questions d’entretien <a name='questions'></a>

**Q : Expliquer le pattern ReAct.**
> ReAct alterne entre Thought (reflexion interne) et Action (appel d outil). Le LLM genere une pensee expliquant son raisonnement, puis un appel d outil structure, recoit l Observation, et repete jusqu a avoir la reponse finale. Simple, efficace, facilement debuggable.

**Q : Quelle est la difference entre MCP, A2A et AG-UI ?**
> MCP (Anthropic) : connecte un LLM a des outils/ressources (USB-C des LLMs). A2A (Google) : communication standardisee entre agents de differents fournisseurs (collaboration inter-agents). AG-UI (CopilotKit) : streaming des evenements d un agent vers une interface utilisateur (transparence et interactivite).

**Q : Comment eviter les boucles infinies dans un agent ?**
> (1) Limiter le nombre de steps (max_steps=10-20) ; (2) Timeout global sur l execution ; (3) Detecter les appels d outils identiques consecutifs (boucle de repetition) ; (4) Budget de tokens maximal par run.

**Q : Qu'est-ce que le KV-Cache dans le contexte des agents ?**
> Dans les agents a longues conversations, le KV-Cache permet de ne pas recalculer les activations pour les tokens precedents. Critique pour les agents avec de longs historiques. Certains frameworks (Anthropic) supportent le prompt caching pour reutiliser le cache entre les appels.

**Q : Comment concevoir un systeme multi-agents robuste ?**
> (1) Definir des interfaces claires (Agent Cards A2A, Tool schemas JSON) ; (2) Chaque agent a une responsabilite unique (single responsibility) ; (3) Communication asynchrone avec une file de taches ; (4) Circuit breakers pour les agents defaillants ; (5) Observabilite : logger tous les appels et resultats ; (6) Evaluation end-to-end avec des cas de test.

**Q : Quels sont les risques de securite specifiques aux agents ?**
> (1) Prompt injection : un document charge par l agent contient des instructions malveillantes. (2) Tool misuse : l agent appelle un outil destructif (DELETE, DROP TABLE). (3) Data exfiltration : l agent envoie des donnees sensibles vers un outil externe. Mitigations : sandboxing des outils, validation des inputs/outputs, RBAC, dry-run mode pour les actions irreversibles.

**Q : Expliquer le concept d'Agent Card dans A2A.**
> Un Agent Card est un fichier JSON (generalement a /.well-known/agent.json) qui decrit publiquement un agent : son nom, ses capacites, ses skills, l URL de son endpoint, ses formats supportes. C est l equivalent d un manifeste d API pour les agents, permettant a d autres agents de decouvrir et utiliser ses services de facon standardisee.

**Q : Quelle est la difference entre la memoire court terme et long terme d un agent ?**
> Court terme (in-context) : l historique de la conversation dans la fenetre de contexte du LLM. Limite par la context window. Long terme (externe) : base de donnees, VDB, SQL persiste entre les sessions. L agent doit explicitement lire/ecrire. Episodique : logs des experiences passees qui peuvent etre recuperes par similarite.
